# Phase 0 — Setup & data loading (FD001)

Deliverable: three clean Python objects — `train_df`, `test_df`, `rul_true` — verified and ready to use for the rest of the project.

See `doc/plan.md` Phase 0 and `doc/requirements_spec.md` Section 3 for the full spec.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")

## 1. Parse train/test into DataFrames

Raw files are space-delimited, no header. Columns: `engine_id, cycle, op_1, op_2, op_3, sensor_1..sensor_21` (26 columns total).

In [2]:
COLUMNS = (
    ["engine_id", "cycle", "op_1", "op_2", "op_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)


def load_fd001(filename: str) -> pd.DataFrame:
    df = pd.read_csv(
        DATA_DIR / filename,
        sep=r"\s+",
        header=None,
        names=COLUMNS,
    )
    df["engine_id"] = df["engine_id"].astype(int)
    df["cycle"] = df["cycle"].astype(int)
    return df


train_df = load_fd001("train_FD001.txt")
test_df = load_fd001("test_FD001.txt")

train_df.head()

,engine_id,cycle,op_1,op_2,op_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [3]:
rul_true = pd.read_csv(
    DATA_DIR / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["rul"],
)["rul"]
rul_true.index = rul_true.index + 1
rul_true.index.name = "engine_id"

rul_true.head()

engine_id
1    112
2     98
3     69
4     82
5     91
Name: rul, dtype: int64

## 2. Check for missing values

In [4]:
train_missing = train_df.isna().sum()
test_missing = test_df.isna().sum()
rul_missing = rul_true.isna().sum()

print("train_df missing values (total):", train_missing.sum())
print("test_df missing values (total):", test_missing.sum())
print("rul_true missing values:", rul_missing)

assert train_missing.sum() == 0, "train_df has missing values"
assert test_missing.sum() == 0, "test_df has missing values"
assert rul_missing == 0, "rul_true has missing values"
print("\nNo missing values in any of the three objects.")

train_df missing values (total): 0
test_df missing values (total): 0
rul_true missing values: 0

No missing values in any of the three objects.


## 3. Min/max cycle count per training engine

In [5]:
cycles_per_train_engine = train_df.groupby("engine_id")["cycle"].max()

print("Training engine lifetimes (max cycle number):")
print(f"  min: {cycles_per_train_engine.min()} cycles (engine {cycles_per_train_engine.idxmin()})")
print(f"  max: {cycles_per_train_engine.max()} cycles (engine {cycles_per_train_engine.idxmax()})")
print(f"  mean: {cycles_per_train_engine.mean():.1f} cycles")

cycles_per_train_engine.describe()

Training engine lifetimes (max cycle number):
  min: 128 cycles (engine 39)
  max: 362 cycles (engine 69)
  mean: 206.3 cycles


count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: cycle, dtype: float64

## 4. Confirm exactly 100 training engines & 100 test engines

In [6]:
n_train_engines = train_df["engine_id"].nunique()
n_test_engines = test_df["engine_id"].nunique()
n_rul_rows = len(rul_true)

print(f"train engines: {n_train_engines}")
print(f"test engines:  {n_test_engines}")
print(f"rul_true rows: {n_rul_rows}")

assert n_train_engines == 100, f"expected 100 training engines, got {n_train_engines}"
assert n_test_engines == 100, f"expected 100 test engines, got {n_test_engines}"
assert n_rul_rows == 100, f"expected 100 RUL values, got {n_rul_rows}"
assert set(train_df["engine_id"]) == set(range(1, 101))
assert set(test_df["engine_id"]) == set(range(1, 101))
assert set(rul_true.index) == set(range(1, 101))
print("\nConfirmed: 100 training engines, 100 test engines, 100 RUL labels.")

train engines: 100
test engines:  100
rul_true rows: 100

Confirmed: 100 training engines, 100 test engines, 100 RUL labels.


## 5. Summary

In [7]:
print("train_df:", train_df.shape)
print("test_df: ", test_df.shape)
print("rul_true:", rul_true.shape)

print("\ndtypes (train_df):")
print(train_df.dtypes)

train_df: (20631, 26)
test_df:  (13096, 26)
rul_true: (100,)

dtypes (train_df):
engine_id      int64
cycle          int64
op_1         float64
op_2         float64
op_3         float64
sensor_1     float64
sensor_2     float64
sensor_3     float64
sensor_4     float64
sensor_5     float64
sensor_6     float64
sensor_7     float64
sensor_8     float64
sensor_9     float64
sensor_10    float64
sensor_11    float64
sensor_12    float64
sensor_13    float64
sensor_14    float64
sensor_15    float64
sensor_16    float64
sensor_17      int64
sensor_18      int64
sensor_19    float64
sensor_20    float64
sensor_21    float64
dtype: object


In [8]:
key_cols = ["cycle", "op_1", "op_2", "op_3", "sensor_1", "sensor_4", "sensor_11"]

print("train_df key column ranges:")
print(train_df[key_cols].agg(["min", "max"]))

print("\ntest_df key column ranges:")
print(test_df[key_cols].agg(["min", "max"]))

print("\nrul_true range: min =", rul_true.min(), ", max =", rul_true.max())

train_df key column ranges:
     cycle    op_1    op_2   op_3  sensor_1  sensor_4  sensor_11
min      1 -0.0087 -0.0006  100.0    518.67   1382.25      46.85
max    362  0.0087  0.0006  100.0    518.67   1441.49      48.53

test_df key column ranges:
     cycle    op_1    op_2   op_3  sensor_1  sensor_4  sensor_11
min      1 -0.0082 -0.0006  100.0    518.67   1384.39      46.80
max    303  0.0078  0.0007  100.0    518.67   1433.36      48.26

rul_true range: min = 7 , max = 145
